<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Barbados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import pandas as pd
from pathlib import Path

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [5]:
DATA_DIR='/content/drive/MyDrive/Barbados'

print(f"Data directory: {DATA_DIR}")

Data directory: /content/drive/MyDrive/Barbados


In [6]:
if os.path.exists(DATA_DIR):
    contents=os.listdir(DATA_DIR)
    print(f"\nContents of Barbados folder:")

    folders=[]
    files=[]

    for item in sorted(contents):
      item_path=os.path.join(DATA_DIR, item)
      if os.path.isdir(item_path):
        folders.append(item)

        try:
          video_count=len([f for f in os.listdir(item_path) if f.endswith('.mp4')])
          print(f" {item:20s} ({video_count} videos)")
        except:
          print(f"{item}")
      else:
        files.append(item)
        file_size=os.path.getsize(item_path)/(1024*1024)
        print(f"{item:20s} ({file_size:.2f} MB)")

    print(f"\nSummary")
    print(f" -{len(folders)} folders")
    print(f" -{len(files)} files")


Contents of Barbados folder:
TestInputSegments.csv (0.82 MB)
Train(1).csv         (4.87 MB)
 normanniles1         (4633 videos)
 normanniles2         (4633 videos)
 normanniles3         (4633 videos)
 normanniles4         (4633 videos)

Summary
 -4 folders
 -2 files


In [11]:
train_csv_paths=[os.path.join(DATA_DIR, 'Train(1).csv')]

TRAIN_CSV=None
for path in train_csv_paths:
    if os.path.exists(path):
        TRAIN_CSV=path
        break

if TRAIN_CSV:
    print(f"Training CSV found: {os.path.basename(TRAIN_CSV)}")

    train_df=pd.read_csv(TRAIN_CSV)
    print(f" -Shape: {train_df.shape}")
    print(f" -Columns: {list(train_df.columns)}")
    print(f"\n First few rows:")
    print(train_df.head(3))
else:
    print("Training CSV not found!")
    print("Looking for: Train(1).csv")

test_csv_paths=[
    os.path.join(DATA_DIR, 'TestInputSegments.csv')
]

TEST_CSV = None
for path in test_csv_paths:
    if os.path.exists(path):
        TEST_CSV = path
        break

if TEST_CSV:
    print(f"\nTest CSV found: {os.path.basename(TEST_CSV)}")

    test_df=pd.read_csv(TEST_CSV)
    print(f" -Shape: {test_df.shape}")
    print(f" -Columns: {list(test_df.columns)}")
    print(f"\n First few rows:")
    print(test_df.head(3))
else:
    print("\nTest CSV not found!")

Training CSV found: Train(1).csv
 -Shape: (16076, 14)
 -Columns: ['responseId', 'view_label', 'ID_enter', 'ID_exit', 'videos', 'video_time', 'datetimestamp_start', 'datetimestamp_end', 'date', 'signaling', 'congestion_enter_rating', 'congestion_exit_rating', 'time_segment_id', 'cycle_phase']

 First few rows:
                responseId       view_label  \
0  zYkHaeOdB7XOnvgP3YW5kQs  Norman Niles #1   
1  NYsHaeCRLq-vnvgPjoXZqA0  Norman Niles #1   
2  A40HaYT8KNm7nvgPq8e12AU  Norman Niles #1   

                                            ID_enter  \
0  time_segment_0_Norman Niles #1_congestion_ente...   
1  time_segment_1_Norman Niles #1_congestion_ente...   
2  time_segment_2_Norman Niles #1_congestion_ente...   

                                             ID_exit  \
0  time_segment_0_Norman Niles #1_congestion_exit...   
1  time_segment_1_Norman Niles #1_congestion_exit...   
2  time_segment_2_Norman Niles #1_congestion_exit...   

                                              vide

In [12]:
WORK_DIR="/content/Barbados/traffic_solution"
os.makedirs(WORK_DIR, exist_ok=True)

OUTPUT_DIR="/content/drive/MyDrive/Barbados/traffic_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

config={
    'DATA_DIR':DATA_DIR,
    'TRAIN_CSV':TRAIN_CSV,
    'TEST_CSV':TEST_CSV,
    'WORK_DIR':WORK_DIR,
    'OUTPUT_DIR': OUTPUT_DIR,
    'video_folders':['normanniles1', 'normanniles2', 'normanniles3', 'normanniles4']
}

import json
config_path=os.path.join(WORK_DIR, 'config.json')
with open(config_path, 'w') as f:
  json.dump(config, f, indent=2)

print(f"Configuration saved to: {config_path}")

Working directory: /content/Barbados/traffic_solution
Output directory: /content/drive/MyDrive/Barbados/traffic_output
Configuration saved to: /content/Barbados/traffic_solution/config.json


In [13]:
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
  train_df=pd.read_csv(TRAIN_CSV)

  print("\nTraining data:")
  print(f" -{len(train_df)} samples")
  print(f" -Columns: {', '.join(train_df.columns)}")

  if 'timestamp' in train_df.columns:
    print(f" -Timestamps: {train_df['timestamp'].iloc[0]} to {train_df['timestamp'].iloc[-1]}")

    label_cols=[col for col in train_df.columns if 'congestion' in col.lower() or 'rating' in col.lower()]
    if label_cols:
      print(f"  -Label columns: {', '.join(label_cols)}")
      for col in label_cols:
        print(f"  -{col}: {train_df[col].value_counts().to_dict()}")

  print("\nVideo Folders:")
  for folder_name in ['normanniles1', 'normanniles2', 'normanniles3', 'normanniles4']:
    folder_path=os.path.join(DATA_DIR, folder_name)
    if os.path.exists(folder_path):
      videos=[f for f in os.listdir(folder_path) if f.endswith('.mp4')]
      print(f" -{folder_name}: {len(videos)} videos")
      if videos:
        print(f"  Example: {videos[0]}")



Training data:
 -16076 samples
 -Columns: responseId, view_label, ID_enter, ID_exit, videos, video_time, datetimestamp_start, datetimestamp_end, date, signaling, congestion_enter_rating, congestion_exit_rating, time_segment_id, cycle_phase

Video Folders:
 -normanniles1: 4633 videos
  Example: normanniles1_2025-10-25-12-17-45.mp4
 -normanniles2: 4633 videos
  Example: normanniles2_2025-10-25-11-53-45.mp4
 -normanniles3: 4633 videos
  Example: normanniles3_2025-10-25-11-55-45.mp4
 -normanniles4: 4633 videos
  Example: normanniles4_2025-10-25-11-54-45.mp4
